In [1]:
from transformers import  AutoModelForSequenceClassification,AutoTokenizer,Trainer,TrainingArguments
from datasets import load_dataset

In [3]:
datasets = load_dataset('json',data_files='./train_pair_1w.json',split='train')

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
datasets[0]

{'sentence1': '找一部小时候的动画片', 'sentence2': '求一部小时候的动画片。谢了', 'label': '1'}

In [5]:
datasets = datasets.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2000
    })
})

In [7]:
#需要把数据处理成 [cls] sentence1 [sep] sentence2 [sep] 这样的形式

import torch

tokenizer = AutoTokenizer.from_pretrained('hfl/chinese-macbert-base')

def process_func(examples):
    tokenized_examples = tokenizer(examples['sentence1'],examples['sentence2'],max_length=128,truncation=True)
    tokenized_examples['labels'] = [float(l) for l in examples['label']]
    return tokenized_examples

tokenized_datasets = datasets.map(process_func,batched=True,remove_columns=datasets['train'].column_names)
tokenized_datasets

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [8]:
#创建模型
model = AutoModelForSequenceClassification.from_pretrained('hfl/chinese-macbert-base',num_labels = 1)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [9]:
import evaluate

acc_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')


In [10]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = [int(p>0.5) for p in predictions]
    labels = [int(l) for l in labels]
    acc = acc_metric.compute(predictions = predictions,references = labels)
    f1 = f1_metric.compute(predictions =predictions,references = labels)
    acc.update(f1)
    
    return acc
    

In [11]:
train_args = TrainingArguments(output_dir="./cross_model",      # 输出文件夹
                               per_device_train_batch_size=32,  # 训练时的batch_size
                               per_device_eval_batch_size=32,   # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               eval_strategy="epoch",           # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
eval_use_gather_object

In [12]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"].select(range(100)), 
                  eval_dataset=tokenized_datasets["test"].select(range(100)), 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_13572/3894911162.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.214813,0.640000,0.608696


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.214813,0.640000,0.608696
2,No log,0.173680,0.740000,0.711111


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.214813,0.640000,0.608696
2,No log,0.173680,0.740000,0.711111
3,0.297000,0.160063,0.760000,0.720930


TrainOutput(global_step=12, training_loss=0.2611207701265812, metrics={'train_runtime': 34.5906, 'train_samples_per_second': 8.673, 'train_steps_per_second': 0.347, 'total_flos': 19287100518312.0, 'train_loss': 0.2611207701265812, 'epoch': 3.0})

In [14]:
trainer.evaluate(tokenized_datasets['test'])

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_13572/2747876658.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p>0.5) for p in predictions]


{'eval_loss': 0.167524054646492,
 'eval_accuracy': 0.755,
 'eval_f1': 0.7203196347031964,
 'eval_runtime': 26.5384,
 'eval_samples_per_second': 75.362,
 'eval_steps_per_second': 2.374,
 'epoch': 3.0}

In [15]:
from transformers import  pipeline

model.config.id2label = {0:"不相似",1:"相似"}


In [16]:
pipe = pipeline('text-classification',model =model,tokenizer=tokenizer,device=0)

Device set to use mps:0


In [26]:
result = pipe({"text": "我喜欢北京", "text_pair": "天气如何"},function_to_apply = 'none')

In [27]:
result["label"] = "相似" if result["score"] > 0.5 else "不相似"
result

{'label': '不相似', 'score': 0.49618101119995117}